# 2.9 联邦学习 (Federated Learning)

在数据不离开本地的前提下，多方协作训练共享模型，保护数据隐私。

本节涵盖：
- 横向联邦学习
- 纵向联邦学习
- 联邦微调
- 隐私保护聚合

## 1. 横向联邦学习（FedAvg）

**目的**：样本不同但特征相同的参与方协作训练

**基本原理**：多个参与方持有不同样本但相同特征空间的数据，各方本地训练后上传梯度或参数更新到中心服务器聚合（如FedAvg），服务器下发聚合后的参数，数据不出本地。

**FedAvg算法**：
1. 服务器初始化全局模型参数
2. 每轮：各客户端下载全局参数，本地训练若干步，上传参数
3. 服务器对客户端参数进行加权平均
4. 重复2-3直到收敛

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

torch.manual_seed(42)

class ClientModel(nn.Module):
    def __init__(self, dim=10, hidden=16, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Linear(hidden, n_classes))
    def forward(self, x):
        return self.net(x)

def fed_avg(client_models, weights=None):
    if weights is None:
        weights = [1.0 / len(client_models)] * len(client_models)
    global_state = copy.deepcopy(client_models[0].state_dict())
    for key in global_state:
        global_state[key] = global_state[key] * weights[0]
        for i in range(1, len(client_models)):
            global_state[key] += client_models[i].state_dict()[key] * weights[i]
    return global_state

n_clients = 4
global_model = ClientModel()
client_models = [ClientModel() for _ in range(n_clients)]

client_data = []
for i in range(n_clients):
    X = torch.randn(50, 10) + i * 0.5
    y = (X[:, 0] + X[:, 1] > 0).long()
    client_data.append((X, y))

print('=== Horizontal Federated Learning (FedAvg) ===')
print(f'Clients: {n_clients}, each with 50 samples')

for round_num in range(10):
    for i, model in enumerate(client_models):
        model.load_state_dict(global_model.state_dict())
        optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
        X, y = client_data[i]
        for _ in range(5):
            loss = F.cross_entropy(model(X), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    global_state = fed_avg(client_models)
    global_model.load_state_dict(global_state)

    if (round_num + 1) % 2 == 0:
        accs = []
        for i, (X, y) in enumerate(client_data):
            acc = (global_model(X).argmax(-1) == y).float().mean()
            accs.append(acc.item())
        print(f'Round {round_num+1}: avg_acc={sum(accs)/len(accs):.3f}, per_client={[f"{a:.3f}" for a in accs]}')

print('\nKey: Data never leaves clients. Only model parameters are shared.')

## 2. 纵向联邦学习

**目的**：特征不同但样本重叠的参与方协作

**基本原理**：多个参与方持有同一样本的不同特征，通过加密计算（如同态加密、安全多方计算）在特征维度上协作训练，各方仅看到自己的特征。

**与横向联邦的区别**：
- 横向：各方数据样本不同，特征相同（如不同地区的用户）
- 纵向：各方数据样本相同，特征不同（如银行和电商的同一用户）

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class VerticalFederatedTrainer:
    def __init__(self, n_parties=2, input_dims=[5, 5], hidden=8, output=3):
        self.n_parties = n_parties
        self.local_encoders = [
            nn.Sequential(nn.Linear(d, hidden), nn.ReLU())
            for d in input_dims
        ]
        self.aggregation_layer = nn.Linear(hidden * n_parties, output)
        self.optimizers = [torch.optim.Adam(enc.parameters(), lr=1e-3) for enc in self.local_encoders]
        self.agg_optimizer = torch.optim.Adam(self.aggregation_layer.parameters(), lr=1e-3)

    def forward(self, feature_splits):
        local_embeds = []
        for i, (enc, features) in enumerate(zip(self.local_encoders, feature_splits)):
            local_embeds.append(enc(features))
        combined = torch.cat(local_embeds, dim=-1)
        return self.aggregation_layer(combined)

    def train_step(self, feature_splits, labels):
        for opt in self.optimizers:
            opt.zero_grad()
        self.agg_optimizer.zero_grad()

        logits = self.forward(feature_splits)
        loss = nn.functional.cross_entropy(logits, labels)
        loss.backward()

        for opt in self.optimizers:
            opt.step()
        self.agg_optimizer.step()
        return loss.item()

n_samples = 200
X_party0 = torch.randn(n_samples, 5)
X_party1 = torch.randn(n_samples, 5)
y = (X_party0[:, 0] + X_party1[:, 0] > 0).long()

trainer = VerticalFederatedTrainer(n_parties=2, input_dims=[5, 5])

print('=== Vertical Federated Learning ===')
print(f'Party A: features 0-4, Party B: features 5-9')
print(f'Each party only sees its own features, embeddings are aggregated centrally')

for epoch in range(20):
    loss = trainer.train_step([X_party0, X_party1], y)
    if (epoch + 1) % 5 == 0:
        with torch.no_grad():
            logits = trainer.forward([X_party0, X_party1])
            acc = (logits.argmax(-1) == y).float().mean()
        print(f'Epoch {epoch+1}: loss={loss:.4f}, accuracy={acc:.3f}')

print('\nKey: Party A never sees features 5-9, Party B never sees features 0-4.')
print('Only intermediate embeddings are shared for aggregation.')

## 3. 联邦微调

**目的**：在联邦场景下微调预训练模型

**基本原理**：各参与方在本地数据上对预训练大模型进行微调（通常使用LoRA等PEFT方法），仅上传LoRA权重或梯度进行聚合，大幅降低通信成本。

**FedLoRA优势**：
- 通信量极小：仅传输LoRA参数（原模型的0.1%），而非完整模型参数
- 隐私增强：LoRA参数更难反推原始数据
- 灵活性：各客户端可以使用不同的LoRA配置

In [ ]:
import torch
import torch.nn as nn
import copy

torch.manual_seed(42)

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank=4, alpha=8):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.W.weight.requires_grad = False
        self.A = nn.Parameter(torch.randn(rank, in_dim) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_dim, rank))
        self.scaling = alpha / rank
    def forward(self, x):
        return self.W(x) + self.scaling * (x @ self.A.T @ self.B.T)

class FedLoRAModel(nn.Module):
    def __init__(self, dim=32, hidden=64, n_classes=3, lora_rank=4):
        super().__init__()
        self.layer1 = LoRALayer(dim, hidden, rank=lora_rank)
        self.layer2 = LoRALayer(hidden, n_classes, rank=lora_rank)
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))
    def get_lora_state(self):
        return {k: v.clone() for k, v in self.state_dict().items() if 'A' in k or 'B' in k}
    def set_lora_state(self, state):
        self.load_state_dict(state, strict=False)

n_clients = 4
global_model = FedLoRAModel()
client_models = [FedLoRAModel() for _ in range(n_clients)]

client_data = [(torch.randn(50, 32) + i, torch.randint(0, 3, (50,))) for i in range(n_clients)]

full_params = sum(p.numel() for p in global_model.parameters())
lora_params = sum(p.numel() for n, p in global_model.named_parameters() if 'A' in n or 'B' in n)
print('=== Federated LoRA Fine-Tuning ===')
print(f'Full parameters: {full_params:,}')
print(f'LoRA parameters: {lora_params:,} ({lora_params/full_params*100:.1f}%)')
print(f'Communication savings: {(1 - lora_params/full_params)*100:.1f}%')

for round_num in range(10):
    for i, model in enumerate(client_models):
        lora_state = global_model.get_lora_state()
        model.set_lora_state(lora_state)
        optimizer = torch.optim.Adam(
            [p for n, p in model.named_parameters() if 'A' in n or 'B' in n], lr=1e-3
        )
        X, y = client_data[i]
        for _ in range(5):
            loss = nn.functional.cross_entropy(model(X), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    all_lora_states = [model.get_lora_state() for model in client_models]
    avg_state = {}
    for key in all_lora_states[0]:
        avg_state[key] = sum(s[key] for s in all_lora_states) / len(all_lora_states)
    global_model.set_lora_state(avg_state)

    if (round_num + 1) % 2 == 0:
        accs = []
        for i, (X, y) in enumerate(client_data):
            acc = (global_model(X).argmax(-1) == y).float().mean()
            accs.append(acc.item())
        print(f'Round {round_num+1}: avg_acc={sum(accs)/len(accs):.3f}')

print('\nKey: Only LoRA parameters are communicated, not the full model.')

## 4. 隐私保护聚合

**目的**：防止从梯度更新中推断原始数据

**基本原理**：使用差分隐私（在梯度中添加噪声）、安全聚合（加密后聚合，服务器无法看到单方梯度）等技术防止成员推断攻击和梯度反转攻击。

**主要威胁**：
- **梯度反转攻击**：从梯度反推训练数据
- **成员推断攻击**：判断某样本是否在训练集中

**防御方法**：
- **差分隐私（DP）**：在梯度中添加校准噪声，提供数学可证明的隐私保障
- **安全聚合**：客户端加密梯度，服务器只能看到聚合结果
- **梯度裁剪**：限制梯度范数，减少信息泄露

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class DPFedAvg:
    def __init__(self, model, max_grad_norm=1.0, noise_multiplier=0.1):
        self.model = model
        self.max_grad_norm = max_grad_norm
        self.noise_multiplier = noise_multiplier

    def clip_gradients(self):
        total_norm = torch.nn.utils.clip_grad_norm_(
            self.model.parameters(), self.max_grad_norm
        )
        return total_norm

    def add_noise(self):
        for param in self.model.parameters():
            if param.grad is not None:
                noise = torch.randn_like(param.grad) * self.noise_multiplier * self.max_grad_norm
                param.grad += noise

class SecureAggregation:
    def __init__(self, n_clients):
        self.n_clients = n_clients

    def generate_masks(self, n_params):
        masks = [torch.randn(n_params) for _ in range(self.n_clients)]
        sum_mask = sum(masks)
        pairwise_cancel = [m - sum_mask / self.n_clients for m in masks]
        return masks, pairwise_cancel

    def aggregate(self, masked_updates):
        return sum(masked_updates) / len(masked_updates)

model = nn.Sequential(nn.Linear(10, 16), nn.ReLU(), nn.Linear(16, 3))
dp_trainer = DPFedAvg(model, max_grad_norm=1.0, noise_multiplier=0.1)

X = torch.randn(32, 10)
y = torch.randint(0, 3, (32,))

print('=== Differential Privacy in Federated Learning ===')
original_grads = []
noisy_grads = []

for step in range(5):
    loss = nn.functional.cross_entropy(model(X), y)
    loss.backward()

    orig_norm = sum(p.grad.norm().item() for p in model.parameters() if p.grad is not None)
    original_grads.append(orig_norm)

    dp_trainer.clip_gradients()
    dp_trainer.add_noise()

    noisy_norm = sum(p.grad.norm().item() for p in model.parameters() if p.grad is not None)
    noisy_grads.append(noisy_norm)

    model.zero_grad()
    print(f'Step {step+1}: orig_grad_norm={orig_norm:.3f}, noisy_grad_norm={noisy_norm:.3f}')

print('\n=== Secure Aggregation Demo ===')
n_clients = 4
n_params = 10
secure_agg = SecureAggregation(n_clients)

client_updates = [torch.randn(n_params) * (i + 1) for i in range(n_clients)]
masks, pairwise = secure_agg.generate_masks(n_params)
masked_updates = [u + m for u, m in zip(client_updates, masks)]

true_avg = sum(client_updates) / n_clients
secure_avg = secure_agg.aggregate(masked_updates)

print(f'True average:  {true_avg[:5].tolist()}')
print(f'Secure average: {secure_avg[:5].tolist()}')
print(f'Difference: {(true_avg - secure_avg).norm():.6f}')
print(f'\nServer only sees masked updates, cannot reconstruct individual client data.')